# Imports

In [3]:
import os
import time
import gc
import sys
import numpy as np
import pandas as pd
import psutil
import requests
from PIL import Image
from tqdm.notebook import tqdm

In [4]:
import cv2

In [5]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from transformers import (
    ViTModel, ViTImageProcessor,
    AutoImageProcessor, AutoModel,
    DeiTModel, DeiTImageProcessor,
    CLIPProcessor, CLIPVisionModel, CLIPTextModel,
    BertTokenizer, BertModel, 
    RobertaTokenizer, RobertaModel,
    GPT2Tokenizer, GPT2Model
)

In [6]:
import torch
from torchvision import models

# Configuration

In [7]:
#("Flickr8k", "Flickr30k", "ConceptualCaptions")
CURRENT_DATASET = "Flickr8k" 

# --- DIRECTORY ARCHITECTURE ---
DATA_DIR = os.path.join(os.getcwd(), 'TFE_Data')
DATASETS_DIR = os.path.join(DATA_DIR, 'Datasets')
RAW_DIR = os.path.join(DATA_DIR, 'Features_RAW')
RESULTS_DIR = os.path.join(DATA_DIR, 'Results')

for d in [DATASETS_DIR, RAW_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Environment initialized. Using device: {device}")

Environment initialized. Using device: cuda


# Data Loading

In [8]:
print(f"Loading dataset: {CURRENT_DATASET}...")

df_path = os.path.join(DATASETS_DIR, f"df_{CURRENT_DATASET}.pkl")
if not os.path.exists(df_path):
    raise FileNotFoundError(f"Dataset file not found at {df_path}. Please run data_builder.ipynb first.")

df = pd.read_pickle(df_path)

# Extract inputs for models
IMAGE_PATHS = df['image_path'].tolist()
ALIGNED_CAPTIONS = [caps[0] for caps in df['captions']]

print(f"Ready. Loaded {len(IMAGE_PATHS)} images and {len(ALIGNED_CAPTIONS)} captions into memory.")

Loading dataset: Flickr8k...
Ready. Loaded 8091 images and 8091 captions into memory.


# Utility Functions
## GreenAI Metrics

In [9]:
def send_ntfy_notification(message, title="TFE Pipeline Update"):
    """Sends a push notification via ntfy.sh."""
    try:
        requests.post(
            f"https://ntfy.sh/{NTFY_TOPIC}",
            data=message.encode(encoding='utf-8'),
            headers={"Title": title}
        )
    except Exception as e:
        print(f"Notification failed: {e}")

def measure_memory():
    """Returns the current memory usage of the process in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

def get_size_in_mb(obj):
    """Returns the size of an object in MB."""
    return sys.getsizeof(obj) / (1024 * 1024)



In [10]:
def execute_and_save(modality, model_name, extract_func, data, device):
    print(f"\nProcessing {modality.upper()} with model: {model_name}")
    
    start_time = time.time()
    mem_before = measure_memory()
    
    # Updated: Vision functions will now handle saliency saving internally
    features = extract_func(data, device)
    
    end_time = time.time()
    mem_after = measure_memory()
    
    exec_time = end_time - start_time
    mem_used = max(0, mem_after - mem_before)
    original_size_mb = get_size_in_mb(features)
    original_dim = features.shape[1] if hasattr(features, 'shape') else 0
    
    save_path = os.path.join(RAW_DIR, f"X_{modality}_{model_name}_raw_{CURRENT_DATASET}.npy")
    np.save(save_path, features)
    
    print(f"Extraction complete. Shape: {features.shape}. Time: {exec_time:.2f}s")
    
    del features
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    return {
        "Modality": modality, "Model": model_name, "Original_Dim": original_dim,
        "Exec_Time_s": exec_time, "Memory_Used_MB": mem_used, "Original_Size_MB": original_size_mb
    }

## XAI HeatMaps (GradCam)

In [11]:
def save_saliency(model_name, activations, img_idx):
    """Saves the 2D spatial grid representing model focus."""
    act = activations.detach().cpu()
    
    # CNN Logic (B, C, H, W) -> Average channels
    if len(act.shape) == 4:
        heatmap = torch.mean(act, dim=1).squeeze().numpy()
    # Transformer Logic (B, Tokens, Dim) -> Reshape patches
    elif len(act.shape) == 3:
        tokens = act[:, 1:, :] # Skip CLS
        side = int(np.sqrt(tokens.shape[1]))
        heatmap = torch.mean(tokens, dim=-1).reshape(side, side).numpy()
    else: return

    # Normalize 0-1
    heatmap = np.maximum(heatmap, 0)
    if np.max(heatmap) > 0: heatmap /= np.max(heatmap)

    path = os.path.join(RAW_DIR, f"Saliency_{model_name}_raw_{img_idx}_{CURRENT_DATASET}.npy")
    np.save(path, heatmap)

In [12]:
def extract_vision_features(model, model_name, dataloader, device, target_layer):
    model.eval()
    features = []
    activations = {}

    def hook_fn(module, input, output):
        activations['value'] = output if isinstance(output, torch.Tensor) else output.last_hidden_state
    
    hook = target_layer.register_forward_hook(hook_fn)
    img_idx = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Extracting {model_name}"):
            batch = batch.to(device)
            if hasattr(model, 'get_image_features'):
                outputs = model.get_image_features(pixel_values=batch)
            else:
                outputs = model(batch)
            
            # Global Embedding
            if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
                batch_features = outputs.pooler_output.cpu().numpy()
            elif hasattr(outputs, 'last_hidden_state'):
                batch_features = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            else:
                batch_features = outputs.cpu().numpy()
            features.append(batch_features)

            # Saliency Maps
            batch_acts = activations['value']
            for i in range(batch_acts.size(0)):
                save_saliency(model_name, batch_acts[i:i+1], img_idx)
                img_idx += 1
                
    hook.remove()
    return np.vstack(features)

# Unimodal Models

## CBIR Vision

### Indexing: Embedding Models

In [ ]:
# --- Vision Model Wrappers with Saliency Targets ---

def get_resnet50_embeddings(image_paths, device):
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights).to(device)
    # Target: Last Convolutional Layer
    target = model.layer4[-1] 
    model.fc = nn.Identity()
    
    dataset = ImageDataset(image_paths, transform=weights.transforms())
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)
    return extract_vision_features(model, "resnet50", dataloader, device, target)

def get_mobilenet_v3_embeddings(image_paths, device):
    weights = models.MobileNet_V3_Large_Weights.DEFAULT
    model = models.mobilenet_v3_large(weights=weights).to(device)
    # Target: Final convolutional layer in the feature extractor
    target = model.features[-1]
    model.classifier = nn.Identity()
    
    dataset = ImageDataset(image_paths, transform=weights.transforms())
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)
    return extract_vision_features(model, "mobilenet_v3", dataloader, device, target)

def get_vit_embeddings(image_paths, device):
    processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
    model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device).eval()
    # Target: The very last Encoder layer
    target = model.encoder.layer[-1]
    
    def transform(img): 
        return processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
    
    dataset = ImageDataset(image_paths, transform=transform)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=0)
    return extract_vision_features(model, "vit", dataloader, device, target)

def get_deit_embeddings(image_paths, device):
    processor = AutoImageProcessor.from_pretrained('facebook/deit-base-distilled-patch16-224')
    model = AutoModel.from_pretrained('facebook/deit-base-distilled-patch16-224').to(device).eval()
    # Target: The last layer of the DeiT encoder
    target = model.encoder.layer[-1]
    
    def transform(img): return processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
    dataset = ImageDataset(image_paths, transform=transform)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)
    return extract_vision_features(model, "deit", dataloader, device, target)

def get_clip_vision_embeddings(image_paths, device):
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    model = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
    # Target: The final layer of the vision encoder inside CLIP
    target = model.vision_model.encoder.layers[-1]
    
    def transform(img): return processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
    dataset = ImageDataset(image_paths, transform=transform)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)
    return extract_vision_features(model, "clip_vision", dataloader, device, target)

## Vision Feature Extractions

In [13]:
class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            # 1. Force RGB at the PIL level
            image = Image.open(img_path).convert('RGB')
            
            # 2. Additional safety: check if PIL actually produced 3 channels
            if len(image.getbands()) != 3:
                image = image.convert('RGB')
                
        except Exception:
            # Placeholder for corrupted web-scraped images
            image = Image.new('RGB', (224, 224), (0, 0, 0)) 
            
        if self.transform:
            try:
                image = self.transform(image)
            except Exception:
                # If the transform still fails (rare), return a zero tensor
                return torch.zeros((3, 224, 224))
        return image

# Override the extraction to be more resilient
def extract_vision_features(model, dataloader, device):
    model.eval()
    features = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting Vision"):
            # Ensure batch is 3-channel (B, 3, H, W)
            if batch.shape[1] != 3:
                continue # Skip weird data chunks if any
                
            batch = batch.to(device)
            if hasattr(model, 'get_image_features'):
                outputs = model.get_image_features(pixel_values=batch)
            else:
                outputs = model(batch)
            
            if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
                batch_features = outputs.pooler_output.cpu().numpy()
            elif hasattr(outputs, 'last_hidden_state'):
                batch_features = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            else:
                batch_features = outputs.cpu().numpy()
                
            features.append(batch_features)
    return np.vstack(features)

def get_resnet50_embeddings(image_paths, device):
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights).to(device)
    model.fc = nn.Identity()
    dataset = ImageDataset(image_paths, transform=weights.transforms())
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)
    return extract_vision_features(model, dataloader, device)

def get_mobilenet_v3_embeddings(image_paths, device):
    weights = models.MobileNet_V3_Large_Weights.DEFAULT
    model = models.mobilenet_v3_large(weights=weights).to(device)
    model.classifier = nn.Identity()
    dataset = ImageDataset(image_paths, transform=weights.transforms())
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)
    return extract_vision_features(model, dataloader, device)

def get_vit_embeddings(image_paths, device):
    from transformers import ViTImageProcessor, ViTModel
    processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
    model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device).eval()
    
    def transform(img): 
        return processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
    
    dataset = ImageDataset(image_paths, transform=transform)
    # num_workers=0 is the safest setting for web-scraped datasets
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=0)
    
    return extract_vision_features(model, dataloader, device)

def get_deit_embeddings(image_paths, device):
    processor = AutoImageProcessor.from_pretrained('facebook/deit-base-distilled-patch16-224')
    model = AutoModel.from_pretrained('facebook/deit-base-distilled-patch16-224').to(device)
    def transform(img): return processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
    dataset = ImageDataset(image_paths, transform=transform)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)
    return extract_vision_features(model, dataloader, device)

def get_clip_vision_embeddings(image_paths, device):
    from transformers import CLIPModel, CLIPProcessor
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    def transform(img): return processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
    dataset = ImageDataset(image_paths, transform=transform)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)
    return extract_vision_features(model, dataloader, device)

## T2T Text 

### Indexing: Embedding Models

In [75]:
# Custom BiLSTM Architecture for the RNN Baseline
class BiLSTMEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=300, hidden_dim=384): 
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        # Concatenate forward and backward final hidden states (384 * 2 = 768 dimensions)
        return torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)

In [76]:
def get_rnn_embeddings(texts, device):
    """Extracts RAW semantic features using sequential modeling (BiLSTM)."""
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BiLSTMEncoder(vocab_size=tokenizer.vocab_size).to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts, desc="Extracting RNN (BiLSTM)"):
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)['input_ids'].to(device)
            embeddings.append(model(inputs).squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [77]:
def get_bert_embeddings(texts, device):
    """Extracts RAW semantic features using bidirectional Auto-Encoder (BERT)."""
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased').to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts, desc="Extracting BERT"):
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
            embeddings.append(model(**inputs).pooler_output.squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [78]:
def get_roberta_embeddings(texts, device):
    """Extracts RAW semantic features using optimized Auto-Encoder (RoBERTa)."""
    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    model = RobertaModel.from_pretrained('roberta-base').to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts, desc="Extracting RoBERTa"):
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
            embeddings.append(model(**inputs).pooler_output.squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [79]:
def get_gpt2_embeddings(texts, device):
    """Extracts RAW semantic features using auto-regressive Decoder (GPT-2)."""
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2Model.from_pretrained('gpt2').to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts, desc="Extracting GPT-2"):
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
            # Use the last hidden state of the last token for causal models
            embeddings.append(model(**inputs).last_hidden_state[:, -1, :].squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [80]:
def get_clip_text_embeddings(texts, device):
    """Extracts RAW semantic features using natively multimodal encoder (CLIP Text)."""
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts, desc="Extracting CLIP Text"):
            inputs = processor(text=text, return_tensors="pt", padding=True, truncation=True).to(device)
            embeddings.append(model(**inputs).pooler_output.squeeze().cpu().numpy())
    return model, np.array(embeddings)

## Text Feature Extraction

In [81]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt'
        )
        return {key: val.squeeze(0) for key, val in encoding.items()}

def extract_text_features(model, dataloader, device, feature_type='last_hidden_state_mean'):
    model.eval()
    features = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting Text"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            
            if feature_type == 'last_hidden_state_mean':
                attention_mask = batch['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
                sum_embeddings = torch.sum(outputs.last_hidden_state * attention_mask, 1)
                sum_mask = torch.clamp(attention_mask.sum(1), min=1e-9)
                batch_features = (sum_embeddings / sum_mask).cpu().numpy()
            else:
                batch_features = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            features.append(batch_features)
    return np.vstack(features)

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        output, (hidden, cell) = self.rnn(embedded)
        return hidden[-1]

def get_rnn_embeddings(texts, device):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = SimpleRNN(tokenizer.vocab_size).to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    
    model.eval()
    features = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting RNN"):
            output = model(batch['input_ids'].to(device))
            features.append(output.cpu().numpy())
    return np.vstack(features)

def get_bert_embeddings(texts, device):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return extract_text_features(model, dataloader, device)

def get_roberta_embeddings(texts, device):
    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    model = RobertaModel.from_pretrained('roberta-base').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return extract_text_features(model, dataloader, device)

def get_gpt2_embeddings(texts, device):
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2Model.from_pretrained('gpt2').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return extract_text_features(model, dataloader, device)

def get_clip_text_embeddings(texts, device):
    from transformers import CLIPModel, CLIPProcessor
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    features = []
    model.eval()
    with torch.no_grad():
        batch_size = 32
        for i in tqdm(range(0, len(texts), batch_size), desc="Extracting CLIP Text"):
            batch_texts = texts[i:i+batch_size]
            inputs = processor(text=batch_texts, return_tensors="pt", padding=True, truncation=True).to(device)
            outputs = model.get_text_features(**inputs)
            
            if hasattr(outputs, 'text_embeds'):
                batch_features = outputs.text_embeds.cpu().numpy()
            elif hasattr(outputs, 'pooler_output'):
                batch_features = outputs.pooler_output.cpu().numpy()
            else:
                batch_features = outputs.cpu().numpy()
            features.append(batch_features)
    return np.vstack(features)

### Execution

In [82]:
green_metrics = []

vision_pipeline = {
    "resnet50": get_resnet50_embeddings,
    "mobilenet_v3": get_mobilenet_v3_embeddings,
    "vit": get_vit_embeddings,
    "deit": get_deit_embeddings,
    "clip_vision": get_clip_vision_embeddings
}

text_pipeline = {
    "rnn": get_rnn_embeddings,
    "bert": get_bert_embeddings,
    "roberta": get_roberta_embeddings,
    "gpt2": get_gpt2_embeddings,
    "clip_text": get_clip_text_embeddings
}

send_ntfy_notification(f"Pipeline started for dataset: {CURRENT_DATASET}", "TFE Indexation Started")

print("\n" + "="*40 + "\nINITIATING VISION PIPELINE\n" + "="*40)
for name, func in vision_pipeline.items():
    metrics = execute_and_save("vision", name, func, IMAGE_PATHS, device)
    green_metrics.append(metrics)
    
send_ntfy_notification(f"Vision pipeline completed for {CURRENT_DATASET}.", "TFE Progress Update")

print("\n" + "="*40 + "\nINITIATING TEXT PIPELINE\n" + "="*40)
for name, func in text_pipeline.items():
    metrics = execute_and_save("text", name, func, ALIGNED_CAPTIONS, device)
    green_metrics.append(metrics)
    
# Save consolidated metrics
df_green = pd.DataFrame(green_metrics)
green_metrics_path = os.path.join(RESULTS_DIR, f'unimodal_green_metrics_RAW_{CURRENT_DATASET}.pkl')
df_green.to_pickle(green_metrics_path)

send_ntfy_notification(
    f"SUCCESS: Full unimodal indexation finished for {CURRENT_DATASET}. Metrics saved.", 
    "TFE Pipeline Finished"
)

print(f"\nPipeline successfully completed. Green Metrics saved to {green_metrics_path}")
display(df_green)

Notification failed: name 'NTFY_TOPIC' is not defined

INITIATING VISION PIPELINE

Processing VISION with model: resnet50


Extracting Vision:   0%|          | 0/994 [00:00<?, ?it/s]

Extraction complete. Shape: (31783, 2048). Time: 30.08s, Memory Delta: 441.77 MB
Saved RAW features to: /home/aysel/tfe/TFE_Data/Features_RAW/X_vision_resnet50_raw_Flickr30k.npy

Processing VISION with model: mobilenet_v3


Extracting Vision:   0%|          | 0/994 [00:00<?, ?it/s]

Extraction complete. Shape: (31783, 960). Time: 28.50s, Memory Delta: 116.46 MB
Saved RAW features to: /home/aysel/tfe/TFE_Data/Features_RAW/X_vision_mobilenet_v3_raw_Flickr30k.npy

Processing VISION with model: vit


...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting CLIP Text:   0%|          | 0/994 [00:00<?, ?it/s]

Extraction complete. Shape: (31783, 512). Time: 15.25s, Memory Delta: 71.88 MB
Saved RAW features to: /home/aysel/tfe/TFE_Data/Features_RAW/X_text_clip_text_raw_Flickr30k.npy
Notification failed: name 'NTFY_TOPIC' is not defined

Pipeline successfully completed. Green Metrics saved to /home/aysel/tfe/TFE_Data/Results/unimodal_green_metrics_RAW_Flickr30k.pkl


,Modality,Model,Original_Dim,Exec_Time_s,Memory_Used_MB,Original_Size_MB
0,vision,resnet50,2048,30.084877,441.773438,248.304810
1,vision,mobilenet_v3,960,28.495012,116.457031,116.392944
2,vision,vit,768,172.905545,93.136719,93.114380
3,vision,deit,768,90.386489,93.175781,93.114380
4,vision,clip_vision,512,41.725331,71.191406,62.076294
5,text,rnn,512,8.710781,62.078125,62.076294
6,text,bert,768,55.685534,93.125000,93.114380
7,text,roberta,768,53.246269,106.476562,93.114380
8,text,gpt2,768,59.767777,103.113281,93.114380
9,text,clip_text,512,15.252379,71.875000,62.076294


In [83]:
# Clean up duplicate runs by keeping only the last successful extraction for each model
df_green = df_green.drop_duplicates(subset=['Modality', 'Model'], keep='last').reset_index(drop=True)

# Re-save the clean table
df_green.to_pickle(green_metrics_path)

print("Cleaned Green Metrics (10 Models):")
display(df_green)

Cleaned Green Metrics (10 Models):


,Modality,Model,Original_Dim,Exec_Time_s,Memory_Used_MB,Original_Size_MB
0,vision,resnet50,2048,30.084877,441.773438,248.304810
1,vision,mobilenet_v3,960,28.495012,116.457031,116.392944
2,vision,vit,768,172.905545,93.136719,93.114380
3,vision,deit,768,90.386489,93.175781,93.114380
4,vision,clip_vision,512,41.725331,71.191406,62.076294
5,text,rnn,512,8.710781,62.078125,62.076294
6,text,bert,768,55.685534,93.125000,93.114380
7,text,roberta,768,53.246269,106.476562,93.114380
8,text,gpt2,768,59.767777,103.113281,93.114380
9,text,clip_text,512,15.252379,71.875000,62.076294
